In [ ]:
import torch

if torch.cuda.is_available():
    print(f"Success! Connected to GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")
else:
    print("GPU not detected. Please check 'Change runtime type' again.")

Success! Connected to GPU: Tesla T4
VRAM Available: 14.6 GB


In [ ]:
# This lists all folders in your Drive so we can find the right name
!ls "/content/drive/MyDrive/"

'12th grade with Physics, Chemistry, and Mathemati... (1).gsheet'
'12th grade with Physics, Chemistry, and Mathemati....gsheet'
'After completing 12th grade with the PCB (Physics... (1).gsheet'
'After completing 12th grade with the PCB (Physics....gsheet'
'Allen Handbook Chemistry .pdf'
'Allen Handbook Mathematics.pdf'
'Allen Handbook Physics .pdf'
'All naming reaction of chemistry Knowledge Hunt  .pdf'
'All organic chemistry reactions in one place 🌚.pdf'
 BU24BTCN1023767.pdf
'can you make pdf of this sheet (1).gsheet'
'can you make pdf of this sheet.gsheet'
'Chart Data Extraction Tool Comparison (1).gsheet'
'Chart Data Extraction Tool Comparison.gsheet'
'ChartSumm Data'
 chetan.pdf
'Colab Notebooks'
'Copy of Gradient Texture UI Morph Slides for Business.gslides'
'Copy of Landscape Morph Picture Slides (1).gslides'
'Copy of Landscape Morph Picture Slides (2).gslides'
'Copy of Landscape Morph Picture Slides.gslides'
'Copy of Priority A Chapters PYQs_Eduniti.pdf'
'Copy of Social Skills L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Correct path based on your screenshots:
# Folder: SpectraProject
# File: Spectra_AI_Project.zip
!unzip -q "/content/drive/MyDrive/SpectraProject/Spectra_AI_Project.zip" -d "/content/dataset/"

# Verify the unzip worked
import os
if os.path.exists("/content/dataset/chart images"):
    print("Success! Dataset is unzipped and ready.")
else:
    print("Files not found. Please check if the zip contains a different internal folder structure.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files not found. Please check if the zip contains a different internal folder structure.


In [ ]:
import os

# This lists every folder in your 'SpectraProject' directory
search_path = "/content/drive/MyDrive/SpectraProject/"

if os.path.exists(search_path):
    print(f"Contents of {search_path}:")
    print(os.listdir(search_path))
else:
    print("The folder 'SpectraProject' was not found. Let's look at your main Drive:")
    print(os.listdir("/content/drive/MyDrive/"))

Contents of /content/drive/MyDrive/SpectraProject/:
['Spectra_AI_Project.zip']


In [ ]:
import os

# 1. Define the exact paths we found
DRIVE_ZIP = "/content/drive/MyDrive/SpectraProject/Spectra_AI_Project.zip"
LOCAL_ZIP = "/content/Spectra_AI_Project.zip"
EXTRACT_PATH = "/content/dataset"

# 2. Copy the file to local disk (Safe for RAM)
print("Copying zip to local storage... please wait.")
!cp "{DRIVE_ZIP}" "{LOCAL_ZIP}"

# 3. Unzip the file
print("Unzipping...")
if not os.path.exists(EXTRACT_PATH):
    os.makedirs(EXTRACT_PATH)

# The '-o' flag forces overwrite without asking
!unzip -o -q "{LOCAL_ZIP}" -d "{EXTRACT_PATH}"

# 4. Clean up the zip to save disk space
!rm "{LOCAL_ZIP}"

print("Unzip Complete! Let's see the folder structure:")
!ls -R {EXTRACT_PATH} | head -n 20

Copying zip to local storage... please wait.
Unzipping...
Unzip Complete! Let's see the folder structure:
/content/dataset:
Spectra_AI_Project

/content/dataset/Spectra_AI_Project:
chart images
ChartSumm Data
ChartSumm-main

/content/dataset/Spectra_AI_Project/chart images:
test_k_0.png
test_k_1000.png
test_k_1001.png
test_k_1002.png
test_k_1003.png
test_k_1004.png
test_k_1005.png
test_k_1006.png
test_k_1007.png
test_k_1008.png
test_k_1009.png


In [ ]:
import json
import os

BASE_DIR = "/content/dataset/Spectra_AI_Project/"
IMAGE_DIR = os.path.join(BASE_DIR, "chart images/")
JSON_INPUT = os.path.join(BASE_DIR, "ChartSumm Data/train_k.json")
OUTPUT_FILE = "/content/dataset/metadata.jsonl"

print("Loading JSON...")
with open(JSON_INPUT, 'r') as f:
    data = json.load(f)

print("\n--- Here is what the first entry in your JSON looks like ---")
print(json.dumps(data[0], indent=2))
print("------------------------------------------------------------\n")

valid_count = 0
with open(OUTPUT_FILE, 'w') as f_out:
    for item in data[:50]: # Still using 50 for the Overfitting Test
        # The script now checks multiple possible key names
        img_name = item.get('image_path') or item.get('image_id') or item.get('filename')
        summary = item.get('summary') or item.get('caption') or item.get('text')

        if img_name and summary:
            # Check if file exists
            if os.path.exists(os.path.join(IMAGE_DIR, img_name)):
                line = {
                    "file_name": img_name,
                    "ground_truth": json.dumps({"gt_parse": {"text_sequence": summary}})
                }
                f_out.write(json.dumps(line) + "\n")
                valid_count += 1

print(f"Successfully mapped {valid_count} images to metadata.jsonl!")

if valid_count == 0:
    print("\nWARNING: Still 0 images mapped. We need to look at the JSON output above to fix the keys.")
else:
    print("\nSUCCESS! You can now re-run the Training block and the error will be gone.")

Loading JSON...

--- Here is what the first entry in your JSON looks like ---
{
  "x_label": "Year",
  "y_label": [
    "Adjusted net intake rate for primary education"
  ],
  "data": {
    "Year": [
      1974,
      1993
    ],
    "Adjusted net intake rate for primary education": [
      23.5892696380615,
      19.8518695831299
    ]
  },
  "title": "Afghanistan - Adjusted net intake rate for primary education",
  "summary": "Afghanistan adjusted net intake rate for primary education was 19.9 % in 1993 , down by 0.00 % from 1974 .",
  "image": "train_k_0.png"
}
------------------------------------------------------------

Successfully mapped 0 images to metadata.jsonl!



In [ ]:
import json
import os
from tqdm import tqdm

# --- EXACT PATHS ---
BASE_DIR = "/content/dataset/Spectra_AI_Project/"
IMAGE_DIR = os.path.join(BASE_DIR, "chart images/")
JSON_INPUT = os.path.join(BASE_DIR, "ChartSumm Data/train_k.json")
OUTPUT_FILE = "/content/dataset/metadata.jsonl"

def prepare_donut_metadata(limit=50):
    with open(JSON_INPUT, 'r') as f:
        data = json.load(f)

    valid_count = 0
    with open(OUTPUT_FILE, 'w') as f_out:
        for item in tqdm(data[:limit]):
            # WE FIXED THE KEYS HERE based on your diagnostic output!
            img_name = item.get('image')
            summary = item.get('summary')

            if img_name and summary:
                # Check if file exists locally
                if os.path.exists(os.path.join(IMAGE_DIR, img_name)):
                    line = {
                        "file_name": img_name,
                        "ground_truth": json.dumps({"gt_parse": {"text_sequence": summary}})
                    }
                    f_out.write(json.dumps(line) + "\n")
                    valid_count += 1

    print(f"\nSUCCESS: Mapped {valid_count} images to metadata.jsonl!")

prepare_donut_metadata()

100%|██████████| 50/50 [00:00<00:00, 34239.22it/s]


SUCCESS: Mapped 50 images to metadata.jsonl!


In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

# Load the processor
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")

# Load model in FP16 (Half-Precision) to save 50% VRAM
model = VisionEncoderDecoderModel.from_pretrained(
    "naver-clova-ix/donut-base",
    torch_dtype=torch.float16
).to("cuda")

# Add the task-specific tokens mentioned in your roadmap
new_tokens = ["<s_chartsumm>", "</s_chartsumm>"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Set model configurations for the ChartSumm task
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_chartsumm>")

print("Model and Processor ready! VRAM usage is optimized.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

The image processor of type `DonutImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/518 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/809M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/809M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model and Processor ready! VRAM usage is optimized.


In [ ]:
import json
import os
import torch
from PIL import Image
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
from datasets import load_dataset

# 1. Load the perfectly formatted metadata.jsonl
dataset = load_dataset("json", data_files="/content/dataset/metadata.jsonl", split="train")

IMAGE_DIR = "/content/dataset/Spectra_AI_Project/chart images/"

# 2. Transform: Safely load images and process text
def transform_and_tokenize(examples):
    safe_images = []

    # Manually open and convert each image
    for file_name in examples["file_name"]:
        img_path = os.path.join(IMAGE_DIR, file_name)
        safe_images.append(Image.open(img_path).convert("RGB"))

    pixel_values = processor(images=safe_images, return_tensors="pt").pixel_values

    labels = []
    for gt_string in examples["ground_truth"]:
        gt_dict = json.loads(gt_string)
        actual_text = gt_dict["gt_parse"]["text_sequence"]

        prompt = f"<s_chartsumm>{actual_text}</s_chartsumm>"

        tokenized = processor.tokenizer(
            prompt,
            padding="max_length",
            max_length=256,
            truncation=True,
            return_tensors="pt"
        )

        label = tokenized.input_ids.squeeze()
        label[label == processor.tokenizer.pad_token_id] = -100
        labels.append(label)

    return {"pixel_values": pixel_values, "labels": torch.stack(labels)}

dataset.set_transform(transform_and_tokenize)

# 3. Define VRAM-Safe Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./spectra_test",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=30,
    logging_steps=5,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
)

# 4. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=processor,
    data_collator=default_data_collator,
)

# 5. START TRAINING
print("Starting Overfitting Test. Training in progress...")
trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

Starting Overfitting Test. Training in progress...


`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


ValueError: Attempting to unscale FP16 gradients.

In [ ]:
!pip install -q transformers datasets sentencepiece pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 65.3 MB/s eta 0:00:00


In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base").to("cuda")

new_tokens = ["<s_chartsumm>", "</s_chartsumm>"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_chartsumm>")

print("Model loaded successfully in 32-bit!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

The image processor of type `DonutImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/518 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/809M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/809M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded successfully in 32-bit!


In [ ]:
import json
import os
import torch
from PIL import Image
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/dataset/metadata.jsonl", split="train")
IMAGE_DIR = "/content/dataset/Spectra_AI_Project/chart images/"

def transform_and_tokenize(examples):
    safe_images = []
    for file_name in examples["file_name"]:
        img_path = os.path.join(IMAGE_DIR, file_name)
        safe_images.append(Image.open(img_path).convert("RGB"))

    pixel_values = processor(images=safe_images, return_tensors="pt").pixel_values

    labels = []
    for gt_string in examples["ground_truth"]:
        gt_dict = json.loads(gt_string)
        actual_text = gt_dict["gt_parse"]["text_sequence"]
        prompt = f"<s_chartsumm>{actual_text}</s_chartsumm>"

        tokenized = processor.tokenizer(
            prompt,
            padding="max_length",
            max_length=256,
            truncation=True,
            return_tensors="pt"
        )

        label = tokenized.input_ids.squeeze()
        label[label == processor.tokenizer.pad_token_id] = -100
        labels.append(label)

    return {"pixel_values": pixel_values, "labels": torch.stack(labels)}

dataset.set_transform(transform_and_tokenize)

training_args = Seq2SeqTrainingArguments(
    output_dir="./spectra_test",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=30,
    logging_steps=5,
    fp16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=processor,
    data_collator=default_data_collator,
)

print("Starting Overfitting Test...")
trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0}.


Starting Overfitting Test...


`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


Step,Training Loss
5,25.782700
10,20.319472
15,16.199449
20,15.148634
25,15.102545
30,11.000208
35,10.802946
40,9.656407
45,9.537920
50,8.466628


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=390, training_loss=4.235549231064625, metrics={'train_runtime': 3162.2092, 'train_samples_per_second': 0.474, 'train_steps_per_second': 0.123, 'total_flos': 1.87614699061248e+19, 'train_loss': 4.235549231064625, 'epoch': 30.0})

In [ ]:
import torch
from PIL import Image

# 1. Pick the first image from your dataset (from your diagnostic earlier)
test_image_path = "/content/dataset/Spectra_AI_Project/chart images/train_k_0.png"
image = Image.open(test_image_path).convert("RGB")

# 2. Prepare the image for the model
pixel_values = processor(image, return_tensors="pt").pixel_values.to("cuda")

# 3. Generate the summary!
print("🧠 Spectra AI is looking at train_k_0.png and generating a summary...\n")

# Set the model to evaluation mode
model.eval()

with torch.no_grad():
    outputs = model.generate(
        pixel_values,
        # Give it the Spectra prompt to start
        decoder_input_ids=processor.tokenizer("<s_chartsumm>", add_special_tokens=False, return_tensors="pt").input_ids.to("cuda"),
        max_length=256,
        early_stopping=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=2, # Uses a slightly smarter guessing algorithm
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

# 4. Decode the AI's output into readable text
decoded_output = processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]

# Clean up the hidden task token for a cleaner print
final_text = decoded_output.replace("<s_chartsumm>", "").strip()

print("--- AI GENERATED SUMMARY ---")
print(final_text)

🧠 Spectra AI is looking at train_k_0.png and generating a summary...

--- AI GENERATED SUMMARY ---
In 2018 , adjusted net intake rate for primary education was 19.9 % in 1993 , down by 0.00 % from 1974 .</s_chartsumm>


In [ ]:
import shutil
from google.colab import drive

# 1. Mount Drive if you haven't already
drive.mount('/content/drive')

# 2. Define your SpectraProject folder path
# Note: Ensure 'SpectraProject' is the exact name in your MyDrive
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SpectraProject/Spectra_Backup_Today"

# 3. Copy the trained model folder to your Drive
shutil.copytree("./spectra_test", DRIVE_PROJECT_PATH, dirs_exist_ok=True)

print(f"✅ SUCCESS! Your trained model is safe at: {DRIVE_PROJECT_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ SUCCESS! Your trained model is safe at: /content/drive/MyDrive/SpectraProject/Spectra_Backup_Today


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Path to your project folder
PROJECT_PATH = "/content/drive/MyDrive/SpectraProject/"
ZIP_FILE = os.path.join(PROJECT_PATH, "Spectra_AI_Project.zip")

# Unzip to local disk
!unzip -q "{ZIP_FILE}" -d /content/dataset/

if os.path.exists("/content/dataset/Spectra_AI_Project/chart images/"):
    print("✅ Success! Images are unzipped.")

Mounted at /content/drive
✅ Success! Images are unzipped.
